# Notebook 06: State Resolution Sensitivity and Cross-Asset Generalization

This notebook performs two robustness analyses:

### Part A: State Resolution Sensitivity (Table T1)
Vary N (number of hidden states) and measure KS pass rate, AD pass rate, and ACF-MAE
to confirm that the model is robust to this hyperparameter.

### Part B: Cross-Asset Generalization (Table T2)
Fit independent CHMM-NJ and CHMM-WJ models to NVDA, JNJ, and JPM to verify
that the framework generalizes beyond SPY.

## Setup

In [ ]:
include("../Include.jl");

In [ ]:
# --- CONSTANTS ---
risk_free_rate = 0.0;
Δt = 1/252;
number_of_paths = 1000;
L = 252;
max_iter = 60;

## Load Data

In [ ]:
# --- LOAD DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_R = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);

# OoS data
oos_dataset = MyOutOfSamplePortfolioDataSet() |> x -> x["dataset"];

# Helper to get returns for a ticker
function get_returns(ticker)
    idx = findfirst(x -> x == ticker, list_of_all_tickers);
    return all_firms_R[:, idx];
end

println("Data loaded: $(length(list_of_all_tickers)) tickers")

## Helper: Full Pipeline for a Single Ticker

In [ ]:
# --- FULL PIPELINE ---
function run_pipeline(R_is, R_oos; K=13, n_paths=1000, max_iter=60,
                      ε=1e-4, λ=100.0, L=252)
    
    n_steps_is = length(R_is);
    n_steps_oos = length(R_oos);
    
    # 1. Train base model
    base = build(MyContinuousHiddenMarkovModel, (
        observations = R_is, number_of_states = K, max_iter = max_iter));
    
    # 2. Build jump model
    jm = build(MyContinuousHiddenMarkovModelWithJumps, (
        base_model = base, epsilon = ε, lambda = λ));
    
    # 3. Stationary distribution
    T_mat = zeros(K, K);
    for i in 1:K; T_mat[i, :] = probs(base.transition[i]); end
    π_stat = (T_mat^1000)[1, :];
    start_dist = Categorical(π_stat);
    
    # 4. Observed metrics
    μ_obs = mean(R_is); σ_obs = std(R_is);
    kurt_obs = sum(((R_is .- μ_obs) ./ σ_obs).^4) / length(R_is) - 3.0;
    acf_obs = autocor(abs.(R_is), 1:L);
    
    # 5. Simulate and evaluate both models
    results = Dict{String, NamedTuple}();
    
    for (name, m) in [("NJ", base), ("WJ", jm)]
        ks_is = 0; ks_oos = 0;
        kurt_sum = 0.0; acf_mae_sum = 0.0;
        kurt_oos_sum = 0.0;
        
        for p in 1:n_paths
            # IS
            s0 = rand(start_dist);
            states_is = m(s0, n_steps_is);
            sim_is = [rand(m.emission[s]) for s in states_is];
            
            pval_is = pvalue(ApproximateTwoSampleKSTest(R_is, sim_is));
            if pval_is > 0.05; ks_is += 1; end
            
            μ_s = mean(sim_is); σ_s = std(sim_is);
            kurt_sum += sum(((sim_is .- μ_s) ./ σ_s).^4) / length(sim_is) - 3.0;
            acf_sim = autocor(abs.(sim_is), 1:L);
            acf_mae_sum += mean(abs.(acf_obs .- acf_sim));
            
            # OoS
            s0 = rand(start_dist);
            states_oos = m(s0, n_steps_oos);
            sim_oos = [rand(m.emission[s]) for s in states_oos];
            
            pval_oos = pvalue(ApproximateTwoSampleKSTest(R_oos, sim_oos));
            if pval_oos > 0.05; ks_oos += 1; end
            
            μ_so = mean(sim_oos); σ_so = std(sim_oos);
            kurt_oos_sum += sum(((sim_oos .- μ_so) ./ σ_so).^4) / length(sim_oos) - 3.0;
        end
        
        results[name] = (
            ks_is = round(100*ks_is/n_paths, digits=1),
            ks_oos = round(100*ks_oos/n_paths, digits=1),
            kurt_is = round(kurt_sum/n_paths, digits=2),
            kurt_oos = round(kurt_oos_sum/n_paths, digits=2),
            acf_mae = round(acf_mae_sum/n_paths, digits=4),
            kurt_obs_is = round(kurt_obs, digits=2)
        );
    end
    
    return results
end

---
## Part A: State Resolution Sensitivity (Table T1)

Vary K ∈ {5, 7, 9, 11, 13, 15, 20} and evaluate KS pass rate and ACF-MAE for SPY.

In [ ]:
# --- STATE RESOLUTION SENSITIVITY ---
K_values = [5, 7, 9, 11, 13, 15, 20];
R_spy = get_returns("SPY");
R_spy_oos = log_growth_matrix(oos_dataset, "SPY"; Δt=Δt, risk_free_rate=risk_free_rate);

println("Table T1: State Resolution Sensitivity (SPY)")
println("="^65)
println("  K  | Model  | KS IS (%) | KS OoS (%) | Kurtosis IS | ACF-MAE")
println("-"^65)

for K in K_values
    results = run_pipeline(R_spy, R_spy_oos; K=K, n_paths=number_of_paths, max_iter=max_iter);
    
    for name in ["NJ", "WJ"]
        r = results[name];
        println("  $(lpad(K,2)) | CHMM-$name | $(lpad(r.ks_is,8)) | $(lpad(r.ks_oos,9)) | $(lpad(r.kurt_is,10)) | $(r.acf_mae)")
    end
end

println("="^65)

---
## Part B: Cross-Asset Generalization (Table T2)

Fit independent models to NVDA, JNJ, and JPM to verify the framework
generalizes across different risk profiles.

In [ ]:
# --- CROSS-ASSET GENERALIZATION ---
cross_tickers = ["NVDA", "JNJ", "JPM"];
K = 13;

println("Table T2: Cross-Asset Generalization (K=$K)")
println("="^80)
println("Ticker | Model  | KS IS (%) | KS OoS (%) | Kurt IS (obs) | Kurt IS (sim) | ACF-MAE")
println("-"^80)

for t in cross_tickers
    R_is = get_returns(t);
    
    # Check if OoS data available
    if !haskey(oos_dataset, t)
        println("$t: OoS data not available, skipping.")
        continue
    end
    R_oos = log_growth_matrix(oos_dataset, t; Δt=Δt, risk_free_rate=risk_free_rate);
    
    results = run_pipeline(R_is, R_oos; K=K, n_paths=number_of_paths, max_iter=max_iter);
    
    for name in ["NJ", "WJ"]
        r = results[name];
        println("$(rpad(t,6)) | CHMM-$name | $(lpad(r.ks_is,8)) | $(lpad(r.ks_oos,9)) | $(lpad(r.kurt_obs_is,12)) | $(lpad(r.kurt_is,12)) | $(r.acf_mae)")
    end
end

println("="^80)

## Cross-Asset ACF Comparison

In [ ]:
# --- CROSS-ASSET ACF VISUALIZATION ---
plots_cross = [];

for t in cross_tickers
    if !haskey(oos_dataset, t); continue; end
    
    R_t = get_returns(t);
    acf_obs = autocor(abs.(R_t), 1:L);
    
    # Train and simulate CHMM-WJ
    base = build(MyContinuousHiddenMarkovModel, (observations=R_t, number_of_states=K, max_iter=max_iter));
    jm = build(MyContinuousHiddenMarkovModelWithJumps, (base_model=base, epsilon=1e-4, lambda=100.0));
    
    T_mat = zeros(K, K);
    for i in 1:K; T_mat[i, :] = probs(base.transition[i]); end
    π_stat = (T_mat^1000)[1, :];
    start_dist = Categorical(π_stat);
    
    acf_sims = hcat([begin
        s0 = rand(start_dist); states = jm(s0, length(R_t));
        sim = [rand(jm.emission[s]) for s in states];
        autocor(abs.(sim), 1:L)
    end for _ in 1:100]...);
    
    acf_mean = mean(acf_sims, dims=2)[:];
    
    p = plot(1:L, acf_obs, lw=2, color=:red, ls=:dash, label="Observed",
        title="$t: ACF(|Gₜ|)", titlefontsize=10,
        xlabel="Lag", ylabel="ACF", legend=:topright);
    plot!(p, 1:L, acf_mean, lw=2, color=:navy, label="CHMM-WJ");
    push!(plots_cross, p);
end

if length(plots_cross) > 0
    fig_cross = plot(plots_cross..., layout=(1, length(plots_cross)), size=(400*length(plots_cross), 350));
    display(fig_cross)
    savefig(fig_cross, joinpath(_PATH_TO_FIGURES, "Fig-Cross-Asset-ACF.svg"))
end

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.